# MLP regression

Joint MIDR/MFA regression using the 14 IM-Corr inputs.

## Configuration

In [ ]:
import os
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import display
from matplotlib import rcParams
from matplotlib.ticker import FuncFormatter, MultipleLocator
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

get_ipython().run_line_magic("matplotlib", "inline")

DATA_PATH = Path("data/Data_clean_english.xlsx")
TARGET_COLS = ["MIDR", "MFA"]
SPLIT_SEED = 42
ARCH_COLS = [
    "stories",
    "first_story_height_m",
    "typical_story_height_m",
    "span_length_m",
    "column_section_dim1_m",
    "fundamental_period_rcf_s",
    "beam_section_dim2_m",
]
PDYBRB_COLS = [
    "apdybrb_fy2_mean",
    "apdybrb_fy2_std",
    "apdybrb_fy2_max",
    "apdybrb_fy2_min",
    "apdybrb_fy2_sum",
    "apdybrb_fy2_var",
    "first_stage_yield_ratio",
    "brace_arrangement",
    "apdybrb_activation_disp_mean",
    "apdybrb_activation_disp_std",
    "apdybrb_activation_disp_max",
    "apdybrb_activation_disp_min",
    "apdybrb_activation_disp_sum",
    "apdybrb_activation_disp_var",
]
CAT_COLS = ["stories", "brace_arrangement", "first_stage_yield_ratio"]
CATEGORY_MAPS = {
    "stories": {5: 0, 7: 1, 9: 2, 11: 3},
    "brace_arrangement": {1: 0, 2: 1, 3: 2},
    "first_stage_yield_ratio": {0.4: 0, 0.5: 1},
}
IM_SCHEMES = {
    "IM-Corr": [
        "SI (cm)",
        "EPV (cm/s)",
        "VSI (cm)",
        "SAavg (g)",
        "PGV (cm/s)",
        "PSV (cm/s)",
        "Sa (g)",
        "Vrms (cm/s)",
        "PGA (g)",
        "ASI (cm/s)",
        "EPA (g)",
        "PSA (g)",
        "Ia (m/s)",
        "Arms (g)",
    ]
}
MODEL_NAME = "MLP"
RESULTS_ROOT = Path(os.environ.get("FWK_RESULTS_ROOT", "results"))
SAVE_DIR = RESULTS_ROOT / MODEL_NAME
FIG_DIR = SAVE_DIR / "figures"
BLAS_THREADS = 1
OMP_THREADS = 1
MODEL_PARAMETERS = {
    "hidden_layer_sizes": (128, 128, 64),
    "activation": "relu",
    "solver": "adam",
    "alpha": 1e-4,
    "learning_rate_init": 1e-3,
    "max_iter": 1500,
    "early_stopping": False,
    "n_iter_no_change": 1500,
    "random_state": 42,
}

## Data preparation

In [ ]:
data = pd.read_excel(DATA_PATH)
data.columns = data.columns.astype(str).str.strip()
ratio = pd.to_numeric(
    data["first_stage_yield_ratio"]
    .astype(str)
    .str.strip()
    .str.replace("%", "", regex=False)
)
data["first_stage_yield_ratio"] = np.round(np.where(ratio > 1, ratio / 100, ratio), 6)
for column in ["stories", "brace_arrangement"]:
    data[column] = pd.to_numeric(data[column], errors="raise")

scheme_name = "IM-Corr"
input_cols = ARCH_COLS + PDYBRB_COLS + IM_SCHEMES[scheme_name]
numeric_cols = [column for column in input_cols if column not in CAT_COLS]
numeric = data[numeric_cols].to_numpy(dtype=float)
y_raw = data[TARGET_COLS].to_numpy(dtype=float)
if not np.isfinite(numeric).all() or not np.isfinite(y_raw).all():
    raise ValueError("Inputs or targets contain NaN/Inf.")

category_codes = []
for column in CAT_COLS:
    codes = data[column].map(CATEGORY_MAPS[column])
    if codes.isna().any():
        raise ValueError(f"Unknown or missing categories in {column}.")
    category_codes.append(codes.to_numpy(dtype=np.float32))
categorical = np.column_stack(category_codes)
feature_names = numeric_cols + CAT_COLS

test2_mask = np.zeros(len(data), dtype=bool)
for story, pga in [(5, 0.5), (7, 0.4), (9, 0.3), (11, 0.2)]:
    test2_mask |= (data["stories"].to_numpy() == story) & np.isclose(
        data["PGA (g)"].to_numpy(), pga, rtol=0, atol=1e-6
    )
train_idx, test1_idx = train_test_split(
    np.flatnonzero(~test2_mask), test_size=0.3, random_state=SPLIT_SEED
)
split_indices = {
    "Train": train_idx,
    "Test1": test1_idx,
    "Test2": np.flatnonzero(test2_mask),
}
scaler_x = StandardScaler().fit(numeric[train_idx])
scaler_y = StandardScaler().fit(y_raw[train_idx])
X = np.column_stack(
    [scaler_x.transform(numeric).astype(np.float32), categorical]
).astype(np.float32)
Y = scaler_y.transform(y_raw).astype(np.float32)
splits = {
    name: {"X": X[ids], "Y": Y[ids], "y_raw": y_raw[ids], "sample_id": ids}
    for name, ids in split_indices.items()
}

## Training

In [ ]:
best_model = MLPRegressor(**MODEL_PARAMETERS)
with (
    threadpool_limits(limits=BLAS_THREADS, user_api="blas"),
    threadpool_limits(limits=OMP_THREADS, user_api="openmp"),
):
    started = time.perf_counter()
    best_model.fit(splits["Train"]["X"], splits["Train"]["Y"])
    training_seconds = time.perf_counter() - started

## Prediction and results

In [ ]:
predictions = {}
metric_rows = []
prediction_times = {}
with (
    threadpool_limits(limits=BLAS_THREADS, user_api="blas"),
    threadpool_limits(limits=OMP_THREADS, user_api="openmp"),
):
    for name in ["Test1", "Test2"]:
        block = splits[name]
        started = time.perf_counter()
        prediction_scaled = best_model.predict(block["X"])
        prediction_times[name] = time.perf_counter() - started
        prediction = scaler_y.inverse_transform(prediction_scaled)
        if (
            prediction.shape != block["y_raw"].shape
            or not np.isfinite(prediction).all()
        ):
            raise ValueError(f"Invalid predictions for {name}.")
        table = pd.DataFrame({"sample_id": block["sample_id"]})
        for j, target in enumerate(TARGET_COLS):
            truth, estimate = block["y_raw"][:, j], prediction[:, j]
            table[f"{target}_true"] = truth
            table[f"{target}_pred"] = estimate
            metric_rows.append(
                {
                    "Split": name,
                    "Target": target,
                    "R2": r2_score(truth, estimate),
                    "RMSE": np.sqrt(mean_squared_error(truth, estimate)),
                    "MAE": mean_absolute_error(truth, estimate),
                }
            )
        predictions[name] = table
metrics = pd.DataFrame(metric_rows)
timing = pd.DataFrame(
    [
        {
            "Train_Time_s": training_seconds,
            "Pred_Time_Test1_s": prediction_times["Test1"],
            "Pred_Time_Test2_s": prediction_times["Test2"],
        }
    ]
)
display(metrics, timing)

SAVE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
for name, table in predictions.items():
    table.to_excel(
        SAVE_DIR / f"{name.lower()}_predictions({MODEL_NAME})_{scheme_name}.xlsx",
        index=False,
    )
with pd.ExcelWriter(SAVE_DIR / f"metrics_summary({MODEL_NAME}).xlsx") as writer:
    metrics.to_excel(writer, sheet_name="Metrics", index=False)
    timing.to_excel(writer, sheet_name="Timing", index=False)
history = pd.DataFrame(
    {
        "iteration": np.arange(1, best_model.n_iter_ + 1),
        "training_loss": best_model.loss_curve_,
    }
)
history.to_csv(SAVE_DIR / "training_history.csv", index=False)
split_manifest = pd.DataFrame({"sample_id": np.arange(len(data)), "split": ""})
for name, ids in split_indices.items():
    split_manifest.loc[ids, "split"] = name
split_manifest.to_csv(SAVE_DIR / "sample_splits.csv", index=False)
_ = joblib.dump(
    {
        "model": best_model,
        "scaler_x": scaler_x,
        "scaler_y": scaler_y,
        "numeric_cols": numeric_cols,
        "category_maps": CATEGORY_MAPS,
        "feature_names": feature_names,
    },
    SAVE_DIR / "model_and_preprocessing.joblib",
)

## Prediction figures

In [ ]:
rcParams["font.family"] = "serif"
rcParams["font.serif"] = ["Times New Roman", "DejaVu Serif"]
rcParams["mathtext.fontset"] = "stix"
rcParams["axes.unicode_minus"] = False
for target, limit, step in [("MIDR", 0.09, 0.02), ("MFA", 20, 5)]:
    fig, ax = plt.subplots(figsize=(7, 6))
    for name, color, marker in [("Test1", "green", "o"), ("Test2", "red", "s")]:
        table = predictions[name]
        score = r2_score(table[f"{target}_true"], table[f"{target}_pred"])
        ax.scatter(
            table[f"{target}_true"],
            table[f"{target}_pred"],
            c=color,
            marker=marker,
            s=40,
            alpha=0.6,
            edgecolors="k",
            linewidths=0.5,
            label=rf"{name}: $R^2$ = {score:.3f}",
        )
    ax.plot([0, limit], [0, limit], color="black", linewidth=1.5)
    ax.set(xlim=(0, limit), ylim=(0, limit))
    ax.xaxis.set_major_locator(MultipleLocator(step))
    ax.yaxis.set_major_locator(MultipleLocator(step))
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_formatter(
            FuncFormatter(lambda value, _: "" if abs(value) < 1e-12 else f"{value:g}")
        )
    ax.annotate(
        "0",
        xy=(0, 0),
        xycoords="axes fraction",
        xytext=(-7, -7),
        textcoords="offset points",
        ha="right",
        va="top",
        fontsize=20,
        annotation_clip=False,
    )
    ax.set_xlabel(f"Reference value ({target})", fontsize=22)
    ax.set_ylabel(f"Predicted value ({target})", fontsize=22)
    ax.tick_params(labelsize=20, direction="in", top=True, right=True, length=5)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper left", fontsize=18, framealpha=0.9, edgecolor="gray")
    fig.tight_layout()
    for extension in ["png", "pdf"]:
        fig.savefig(
            FIG_DIR / f"MLP_Comparison_{target}_{scheme_name}.{extension}", dpi=300
        )
    display(fig)
    plt.close(fig)